In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('quikr_car.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Data Cleaning

In [ ]:
backup = df.copy()

In [ ]:
df.shape

In [ ]:
# Removing non-integer values from year column
df = df[df['year'].str.isnumeric()]
df['year']=df['year'].astype(int)
df.shape

In [ ]:
df['Price'].unique

In [ ]:
# Cleaning and type casting price column
df = df[df['Price']!='Ask For Price']
df['Price']=df['Price'].str.replace(',','').astype(int)

In [ ]:
# Typecasting and cleaning kms_driven column
df['kms_driven'] = df['kms_driven'].str.split(' ').str.get(0).str.replace(',','')
df = df[df['kms_driven'].str.isnumeric()]
df['kms_driven'] = df['kms_driven'].astype(int)

In [ ]:
# Removing NA Value rows
df=df[~df['fuel_type'].isna()]

In [ ]:
df['name']=df['name'].str.split(' ').str.slice(0,3).str.join(' ')

In [ ]:
df.info()

In [ ]:
df = df[df['Price']<6e6]

In [ ]:
backup.shape

In [ ]:
df.shape

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.to_csv('car.csv')

## Model

In [ ]:
X = df.drop(columns=['Price'])
y = df['Price']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.1)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [ ]:
ohe = OneHotEncoder()
ohe.fit(X[['name','company','fuel_type']])

In [ ]:
column_trans = make_column_transformer((OneHotEncoder(categories=ohe.categories_),['name','company','fuel_type']),remainder='passthrough')

In [ ]:
lr = LinearRegression()

In [ ]:
pipe=make_pipeline(column_trans,lr)

In [ ]:
pipe.fit(X_train,y_train)

In [ ]:
y_pred = pipe.predict(X_test)

In [ ]:
r2_score(y_test,y_pred)

In [ ]:
score = []
for i in range(0,10000):
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=i)
    lr = LinearRegression()
    pipe = make_pipeline(column_trans,lr)
    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    score.append(r2_score(y_test,y_pred))
    if i%1000 == 0:
        print(f'{i/10000 * 100}%',end='==> ')
print("100%",end='')

In [ ]:
max(score)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=np.argmax(score))
lr = LinearRegression()
pipe = make_pipeline(column_trans,lr)
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
r2_score(y_test,y_pred)

# Pickle Dumping the File

In [ ]:
import pickle
pickle.dump(pipe,open('LinearRegressionModel.pkl','wb'))

In [ ]:
pipe.predict(pd.DataFrame([["Maruti Suzuki Swift","Maruti",2011  ,10000,"Petrol"]],columns=['name','company','year','kms_driven','fuel_type']))